In [ ]:
import os, time, math, copy, random, json
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt
import pandas as pd

QUICK = False          # True -> tiny subset + few epochs, just to check that everything runs

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = device.type == 'cuda'
print('device:', device, '| torch', torch.__version__, '| AMP:', USE_AMP)
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))

def seed_all(seed=0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_all(0)
torch.backends.cudnn.benchmark = True

# ---- experiment sizes (full vs quick) --------------------------------------
CFG = dict(
    n_train   = 45_000 if not QUICK else 2_000,
    n_val     = 5_000  if not QUICK else 1_000,
    ep_t1     = 10 if not QUICK else 1,     # Task 1c MLP epochs
    ep_t2     = 8  if not QUICK else 1,     # Task 2c epochs per (optimizer, lr)
    ep_t3a    = 30 if not QUICK else 1,     # Task 3a epochs (5k subset)
    n_t3a     = 5_000 if not QUICK else 1_000,
    ep_t3b    = 20 if not QUICK else 1,     # Task 3b epochs per network
    depths_3b = [8, 20, 56] if not QUICK else [8, 20],
    ep_t4     = 20 if not QUICK else 1,     # Task 4 (hard limit 20)
    bs        = 128,
)
print(json.dumps(CFG, indent=1))